# D-MTHD tweet benchmark: full run

Before running: right panel -> Session options -> Accelerator **GPU T4 x2**, Internet **On**. Attaching the `andrewmvd/cyberbullying-classification` dataset is optional: if it is not found, the same CSV is downloaded from its Hugging Face mirror.

Then **Save Version -> Save & Run All (Commit)**. If anything fails, this version shows **Failed** and the log ends with the reason.

Second version (comparison grid): add the first version's output as an input, set `RESUME_FROM` in the run cell to its path, Save & Run All again. Finished work is skipped.

In [ ]:
import os, subprocess, glob, shutil, zipfile
REPO = "https://github.com/mahdihasanshadi/THESIS.git"
DEST = "/kaggle/working/dmthd-p3"
if not os.path.exists(os.path.join(DEST, "src", "dmthd")):
    r = subprocess.run(["git", "clone", "-q", REPO, DEST])          # works if the repo is public
    if r.returncode != 0:
        shutil.rmtree(DEST, ignore_errors=True)
        z = glob.glob("/kaggle/input/**/dmthd-p3-code.zip", recursive=True)
        tree = glob.glob("/kaggle/input/**/src/dmthd/train_student.py", recursive=True)
        if z:                                                            # zip attached as-is
            zipfile.ZipFile(z[0]).extractall(DEST)
        elif tree:                                                       # Kaggle auto-extracted the zip
            root = os.path.dirname(os.path.dirname(os.path.dirname(tree[0])))
            shutil.copytree(root, DEST)
        else:
            raise SystemExit("clone failed and no code found among the inputs: attach the code dataset")
os.chdir(DEST)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
print(open("README.md").read()[:400])

In [ ]:
import os, subprocess, glob, sys, urllib.request
env = dict(os.environ, ROOT='/kaggle/working', PYTHONPATH='src', GPU='1', SEEDS='1,2,3',
           COMMITTEES='homo,hetero', MODES='ft,skd,uniform,dmthd',
           TEACHERS='bert-large-uncased:bert-large,GroNLP/hateBERT:hatebert,cardiffnlp/twitter-roberta-base-irony:irony',
           STUDENTS='google/bert_uncased_L-4_H-256_A-4:bert-mini,google/bert_uncased_L-4_H-512_A-8:bert-small,distilbert-base-uncased:distilbert')
env['RESUME_FROM'] = ''   # second version: attach the first version's output as an input and put its path here
raw = (glob.glob('/kaggle/input/**/cyberbullying_tweets.csv', recursive=True) or [None])[0]
print('inputs seen:', glob.glob('/kaggle/input/*'), '| csv from input:', raw, flush=True)
if raw is None:   # dataset not attached: fetch the same file from its Hugging Face mirror (Internet must be on)
    os.makedirs('/kaggle/working/raw', exist_ok=True)
    raw = '/kaggle/working/raw/cyberbullying_tweets.csv'
    urllib.request.urlretrieve('https://huggingface.co/datasets/mattematics/cyberbullying/resolve/main/cyberbullying_tweets.csv', raw)
    print('downloaded corpus from Hugging Face:', os.path.getsize(raw), 'bytes', flush=True)
cmd = ['python', 'kaggle/run_benchmark.py', '--dataset', 'tweets', '--stage', 'all', '--raw', raw]
print('running:', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, env=env)
if r.returncode != 0:
    raise SystemExit(f'BENCHMARK FAILED with exit code {r.returncode}: scroll up in this log to the first Traceback')
print('BENCHMARK FINISHED')


In [ ]:
# Pack everything worth keeping so it can be downloaded from the notebook output
!cd /kaggle/working && tar czf dmthd_runs.tgz runs cache/tweets/meta.json data/tweets/report.json && ls -la dmthd_runs.tgz
!python -m dmthd.aggregate --runs /kaggle/working/runs/tweets